In [3]:
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("CH16-Evaluations")

loader = PDFPlumberLoader("../data/SPRI_AI_Brief_2023년12월호_F.pdf")

docs = loader.load()
docs = docs[3:-1]

len(docs)

LangChain API Key가 설정되지 않았습니다. 참고: https://wikidocs.net/250954


19

In [4]:
docs[0].metadata

{'source': '../data/SPRI_AI_Brief_2023년12월호_F.pdf',
 'file_path': '../data/SPRI_AI_Brief_2023년12월호_F.pdf',
 'page': 3,
 'total_pages': 23,
 'Author': 'dj',
 'Creator': 'Hwp 2018 10.0.0.13462',
 'Producer': 'Hancom PDF 1.3.0.542',
 'CreationDate': "D:20231208132838+09'00'",
 'ModDate': "D:20231208132838+09'00'",
 'PDFVersion': '1.4'}

In [5]:
for doc in docs:
    doc.metadata["filename"] = doc.metadata["source"]

In [8]:
from ragas.testset.generator import TestsetGenerator
from ragas.testset.evolutions import simple, reasoning, multi_context, conditional
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset.extractor import KeyphraseExtractor
from ragas.testset.docstore import InMemoryDocumentStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


generator_llm = ChatOpenAI(model="gpt-4o-mini")
critic_llm = ChatOpenAI(model="gpt-4o-mini")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [9]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

langchain_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))

keyphrase_extractor = KeyphraseExtractor(llm=langchain_llm)

ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

docstore = InMemoryDocumentStore(
    splitter = splitter,
    embeddings = ragas_embeddings,
    extractor = keyphrase_extractor
)

In [10]:
generator = TestsetGenerator.from_langchain(
    generator_llm,
    critic_llm,
    ragas_embeddings,
    docstore=docstore,
)

In [11]:
distributions = {
    simple: 0.4,
    reasoning: 0.2,
    multi_context: 0.2,
    conditional: 0.2
}

In [12]:
testset = generator.generate_with_langchain_docs(
    documents = docs,
    test_size = 10,
    distributions = distributions,
    with_debugging_logs = True,
    raise_exceptions = False,
)

Generating:   0%|          | 0/10 [00:00<?, ?it/s]              [ragas.testset.filters.DEBUG] context scoring: {'clarity': 1, 'depth': 2, 'structure': 2, 'relevance': 3, 'score': 2.0}
[ragas.testset.evolutions.DEBUG] keyphrases in merged node: ['AI Safety Summit', 'AI governance', 'AI risk management', 'AI policy recommendations', 'AI safety measures']
[ragas.testset.filters.DEBUG] context scoring: {'clarity': 1, 'depth': 2, 'structure': 2, 'relevance': 2, 'score': 1.75}
[ragas.testset.evolutions.DEBUG] keyphrases in merged node: ['AI Safety Institute', 'AI governance', 'AI risk management', 'AI development', 'AI safety measures']
[ragas.testset.filters.DEBUG] context scoring: {'clarity': 1, 'depth': 2, 'structure': 1, 'relevance': 2, 'score': 1.5}
[ragas.testset.evolutions.DEBUG] keyphrases in merged node: ['AI systems', 'Advanced AI', 'Data privacy', 'Ethical guidelines', 'International Code of Conduct']
[ragas.testset.filters.DEBUG] context scoring: {'clarity': 1, 'depth': 2, 'struc

In [13]:
test_df = testset.to_pandas()
test_df.head()

,question,contexts,ground_truth,evolution_type,metadata,episode_done
0,What is the significance of AI governance in e...,"[£유튜브, 특정인을 모방한 AI 생성 콘텐츠에 대한 삭제 요청에도 대응 계획\nn...",The significance of AI governance in ensuring ...,simple,[{'source': '../data/SPRI_AI_Brief_2023년12월호_F...,True
1,What are the key aspects of AI regulation disc...,"[SPRi AI Brief |\n2023-12월호\nEU AI 법 3자 협상, 기반...",The key aspects of AI regulation discussed in ...,simple,[{'source': '../data/SPRI_AI_Brief_2023년12월호_F...,True
2,What specific changes in productivity and user...,"[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n빌 게이츠, A...",The context discusses that AI is expected to s...,simple,[{'source': '../data/SPRI_AI_Brief_2023년12월호_F...,True
3,What factors affect neural network AI model pe...,[∙ 또한 크라우드소싱 플랫폼이 할당한 라이선스는 데이터셋 원저작자의 의도보다 더 ...,The context does not provide specific factors ...,reasoning,[{'source': '../data/SPRI_AI_Brief_2023년12월호_F...,True
4,What improvements in Tongyi Qianwen 2.0 boost ...,[1. 정책/법제 2. 기업/산업 3. 기술/연구 4. 인력/교육\n알리바바 클라우...,The improvements in Tongyi Qianwen 2.0 that bo...,reasoning,[{'source': '../data/SPRI_AI_Brief_2023년12월호_F...,True


In [15]:
test_df.to_csv("../data/ragas_synthetic_dataset.csv", index=False)